# Interleaved Multimodal Data — Quickstart

> **What you'll learn**: How NeMo Curator's interleaved multimodal module works —
> its data model, reader / filter / writer stages, and custom filter API —
> using a real MINT-1T PDF shard as the working dataset.

Each MINT-1T PDF is a sequence of page images interleaved with extracted text.
NeMo Curator represents it as a flat `InterleavedBatch` — one row per text block,
image, or metadata entry — shared by every reader, filter, and writer.

| Section | Curator concept introduced |
|---------|---------------------------|
| 0 — Setup | `FileGroupTask` |
| 1 — Load & First Look | `InterleavedWebdatasetReaderStage` · `InterleavedBatch` · `INTERLEAVED_SCHEMA` |
| 2 — Explore | `batch.to_pandas()` · `batch.count()` · `batch.num_items` |
| 3 — Filter Images | `InterleavedAspectRatioFilterStage` (built-in `BaseInterleavedFilterStage`) |
| 4 — Filter Text | Custom `BaseInterleavedFilterStage` subclass · `content_keep_mask` |
| 5 — Save & Verify | `InterleavedParquetWriterStage` · `InterleavedParquetReaderStage` · `schema_overrides` |

---

## Installation

```bash
pip install "nemo_curator[interleaved_cpu]"    # CPU (for CPU only modules)
pip install "nemo_curator[interleaved_cuda12]" # GPU (for CPU+GPU modules)
```

In [ ]:
import json
import os
import tarfile
import tempfile
from dataclasses import dataclass
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
from IPython.display import Image, Markdown
from IPython.display import display as ipy_display
from PIL import Image as PILImage

from nemo_curator.stages.interleaved.io.readers.parquet import InterleavedParquetReaderStage
from nemo_curator.stages.interleaved.io.readers.webdataset import InterleavedWebdatasetReaderStage
from nemo_curator.stages.interleaved.io.writers.tabular import InterleavedParquetWriterStage
from nemo_curator.stages.interleaved.io.writers.webdataset import InterleavedWebdatasetWriterStage
from nemo_curator.stages.interleaved.stages import (
    BaseInterleavedFilterStage,
    InterleavedAspectRatioFilterStage,
)
from nemo_curator.tasks import FileGroupTask
from nemo_curator.tasks.interleaved import INTERLEAVED_SCHEMA, InterleavedBatch

In [ ]:
# Pipeline parameters — thresholds the filter stages use
MIN_RATIO = 0.5  # minimum image aspect ratio (width / height)
MAX_RATIO = 2.0  # maximum image aspect ratio
MIN_CHARS = 50  # minimum text length in characters

In [ ]:
# Display settings — only affect notebook rendering, not the pipeline
MAX_CHARS = 350  # max text chars to show when rendering a sample
THUMB_W = 480  # thumbnail width in pixels
RESAMPLE = getattr(PILImage, "Resampling", PILImage).LANCZOS

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 45)

In [ ]:
def render_sample(df: pd.DataFrame, sample_id: str, max_chars: int = MAX_CHARS, thumb_w: int = THUMB_W) -> None:
    """Render a sample as interleaved text + inline images."""
    sample = df[df["sample_id"] == sample_id].sort_values("position")
    n_img = int((sample["modality"] == "image").sum())
    n_txt = int((sample["modality"] == "text").sum())
    ipy_display(
        Markdown(f"### Document `{sample_id}`  ({n_img + n_txt} content rows: {n_img} images + {n_txt} text blocks)")
    )
    meta = sample[sample["modality"] == "metadata"]
    if not meta.empty:
        r = meta.iloc[0]
        url = r.get("url")
        lang = r.get("language_id_whole_page_fasttext")
        print(f"  url:      {str(url)[:100] if url is not None and not pd.isna(url) else '-'}")
        print(f"  language: {lang if lang is not None and not pd.isna(lang) else '-'}")
        print()
    for _, row in sample[sample["modality"] != "metadata"].iterrows():
        pos = int(row["position"])
        if row["modality"] == "text":
            tc = row.get("text_content")
            text = str(tc) if tc is not None and not pd.isna(tc) else ""
            snippet = text[:max_chars] + " \u2026" if len(text) > max_chars else text
            print(f"[pos {pos}] TEXT  ({len(text)} chars)")
            print(f"  {snippet}")
            print()
        elif row["modality"] == "image":
            bc = row["binary_content"]
            if bc is not None and not pd.isna(bc):
                img = PILImage.open(BytesIO(bytes(bc)))
                w, h = img.size
                ref = json.loads(row["source_ref"]) if isinstance(row.get("source_ref"), str) else {}
                print(f"[pos {pos}] IMAGE  {w}x{h}  frame={ref.get('frame_index', '-')}")
                thumb = img.copy()
                thumb.thumbnail((thumb_w, thumb_w))
                buf = BytesIO()
                thumb.save(buf, format="PNG")
                ipy_display(Image(data=buf.getvalue(), format="png"))
                print()


def render_image_strip(df_sample: pd.DataFrame, label: str, ncols: int = 6) -> None:
    """Show a horizontal strip of thumbnails for the image rows of one sample."""
    img_rows = df_sample[df_sample["modality"] == "image"].sort_values("position")
    ipy_display(Markdown(f"**{label}** \u2014 {len(img_rows)} image rows"))
    if img_rows.empty:
        print("  (no images)")
        return
    n = min(ncols, len(img_rows))
    _fig, axes = plt.subplots(1, n, figsize=(n * 2.4, 2.6))
    axes = [axes] if n == 1 else list(axes)
    for ax, (_, row) in zip(axes, img_rows.iterrows(), strict=False):
        bc = row["binary_content"]
        if bc is not None and not pd.isna(bc):
            img = PILImage.open(BytesIO(bytes(bc)))
            w, h = img.size
            ratio = w / h if h > 0 else 0
            img.thumbnail((128, 128), RESAMPLE)
            ax.imshow(img)
            ax.set_title(f"pos {int(row['position'])}\n{w}x{h}  r={ratio:.2f}", fontsize=7)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def compute_image_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Extract image rows and decode width, height, and aspect ratio from binary_content."""
    rows = df[df["modality"] == "image"].copy().reset_index(drop=True)

    def _image_size(bc: bytes) -> tuple[int | None, int | None]:
        try:
            img = PILImage.open(BytesIO(bytes(bc)))
            return img.size[0], img.size[1]
        except (OSError, ValueError):
            return None, None

    sizes = [_image_size(bc) for bc in rows["binary_content"]]
    widths = [s[0] for s in sizes]
    heights = [s[1] for s in sizes]
    rows["width"] = widths
    rows["height"] = heights
    rows = rows.dropna(subset=["width", "height"])
    rows = rows[rows["height"] > 0].copy()
    rows["ratio"] = (rows["width"] / rows["height"]).round(2)
    return rows

---
## Section 0 — Setup

Download one MINT-1T PDF shard (~79 MB). Set `MINT1T_TAR_PATH` to skip the download.

> **`FileGroupTask`** is the standard input task for all Curator readers. It carries
> a list of file paths — the only thing a reader stage needs to start processing.

In [ ]:
LOCAL_TAR = os.environ.get("MINT1T_TAR_PATH", "")

if not LOCAL_TAR or not Path(LOCAL_TAR).exists():
    print("Downloading MINT-1T shard from HuggingFace (~79 MB)...")
    LOCAL_TAR = hf_hub_download(
        repo_id="mlfoundations/MINT-1T-PDF-CC-2024-18",
        filename="CC-MAIN-2024-18-shard-0/CC-MAIN-20240412101354-20240412131354-00000.tar",
        repo_type="dataset",
    )
print(f"Using: {LOCAL_TAR}")

---
## Section 1 — Load & First Look

`InterleavedWebdatasetReaderStage` reads each MINT-1T sample into an `InterleavedBatch`.
Text and images share one interleaved position sequence in document order:

```
pos -1  →  metadata   url, language_id, pdf_name, …   (one row per sample)
pos  0  →  image      page-1 scan
pos  1  →  text       "INTRODUCTION…"
pos  2  →  image      page-2 scan
pos  3  →  text       "SUMMARY…"
…
```

**`InterleavedBatch`** is the minimum unit of data in Curator's interleaved pipeline —
the object that flows between every reader, filter, and writer stage.
Its data model is defined by `INTERLEAVED_SCHEMA`, eight columns every stage reads and writes:

| Field | Role |
|-------|------|
| `sample_id` | Groups all rows belonging to one document |
| `position` | Document order: `-1` for metadata, `≥ 0` for content (shared by text and image) |
| `modality` | Row type: `"metadata"`, `"text"`, or `"image"` |
| `text_content` | Text string — populated for `"text"` rows |
| `binary_content` | Image bytes — populated for `"image"` rows when materialized |
| `source_ref` | JSON pointer used for lazy image fetch |
| `content_type` | MIME type of the content |
| `materialize_error` | Error message if an image fetch failed |

Any additional fields in the source JSON (url, language, pdf_name, …) are carried as
passthrough columns alongside the schema.

*Inspection helpers* — `batch.num_items` (distinct samples) and `batch.count(modality=...)`
(row count) are useful for quick sanity checks but are not part of the data model.

> **Lazy vs Eager**: `materialize_on_read=False` (default) keeps `binary_content=None`
> and fetches bytes on demand — efficient for large-scale pipelines. We use `True` here
> so images are ready for inline display.

In [ ]:
task = FileGroupTask(dataset_name="mint1t", data=[LOCAL_TAR], _metadata={})

reader = InterleavedWebdatasetReaderStage(
    materialize_on_read=True,  # load all image bytes immediately
    per_image_fields=["image_metadata"],  # image_metadata is per-image, not per-sample
)
result = reader.process(task)
batch = result[0] if isinstance(result, list) else result

print(f"{batch.num_items} samples  |  {batch.count()} total rows")
print(f"  {batch.count(modality='image'):3d} image rows")
print(f"  {batch.count(modality='text'):3d} text rows")
print(f"  {batch.count(modality='metadata'):3d} metadata rows")

In [ ]:
df = batch.to_pandas()

# Truncate large columns for display
display_df = df.copy()
display_df["binary_content"] = display_df["binary_content"].apply(
    lambda b: f"<{len(b):,} bytes>" if b is not None and not pd.isna(b) else None
)
display_df["source_ref"] = display_df["source_ref"].str[:50] + "..."
display_df.head(6)

In [ ]:
reserved = set(INTERLEAVED_SCHEMA.names)
passthrough = [c for c in df.columns if c not in reserved]

print("Reserved columns (managed by the pipeline):")
print(f"  {'Column':<25} {'Type':<18} {'Populated for'}")
print(f"  {'-' * 70}")
notes = {
    "sample_id": "all rows",
    "position": "all rows  (-1 for metadata, >= 0 for content)",
    "modality": "all rows  ('text', 'image', or 'metadata')",
    "content_type": "text and image rows (MIME type)",
    "text_content": "text rows only",
    "binary_content": "image rows when materialized",
    "source_ref": "image rows \u2014 JSON pointer for lazy fetch",
    "materialize_error": "image rows that failed to fetch",
}
for f in INTERLEAVED_SCHEMA:
    print(f"  {f.name:<25} {f.type!s:<18} {notes.get(f.name, '')}")

print(f"\nPassthrough columns from source JSON ({len(passthrough)}): {passthrough}")

---
## Section 2 — Explore: What Does the Data Look Like?

`batch.to_pandas()` converts the `InterleavedBatch` into a flat Pandas DataFrame —
the same table that all subsequent cells inspect and filter.

In [ ]:
rich_sample_id = df["sample_id"].iloc[0]
render_sample(df, rich_sample_id)

### Dataset overview — modality mix per sample

In [ ]:
counts = (
    df[df["modality"].isin(["text", "image"])]
    .pivot_table(index="sample_id", columns="modality", aggfunc="size", fill_value=0)
    .reindex(columns=["text", "image"], fill_value=0)
    .sort_values("image", ascending=False)
)

fig, ax = plt.subplots(figsize=(11, 5))
y = range(len(counts))
ax.barh(y, counts["text"], label="text rows", color="#4C72B0")
ax.barh(y, counts["image"], left=counts["text"], label="image rows", color="#DD8452")
ax.set_xlabel("Number of rows")
ax.set_title(
    f"Modality composition \u2014 {len(counts)} samples  "
    f"(avg {counts['text'].mean():.1f} text + {counts['image'].mean():.1f} images per sample)"
)
ax.legend()
ax.set_yticks([])
plt.tight_layout()
plt.show()

---
## Section 3 — Filter: Image Quality

We filter images by **aspect ratio** (width / height). Images outside [0.5, 2.0]
are typically thin decorative rules, OCR artifacts, or corrupted frames — not useful
training signal.

We look at the distribution *first*, then preview exactly which images will be removed,
*then* apply the filter.

`InterleavedAspectRatioFilterStage` is a built-in subclass of `BaseInterleavedFilterStage`.
After filtering it automatically renumbers positions (0, 1, 2 …) and removes any
metadata rows whose sample lost all its content. Section 4 shows how to write your own
filter using the same mechanism.

In [ ]:
image_rows = compute_image_rows(df)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(image_rows["ratio"].clip(0, 4), bins=50, color="#4C72B0", edgecolor="white", alpha=0.85)

ax.axvline(MIN_RATIO, color="#d62728", linestyle="--", linewidth=2, label=f"min = {MIN_RATIO}")
ax.axvline(MAX_RATIO, color="#d62728", linestyle="--", linewidth=2, label=f"max = {MAX_RATIO}")
xlim = ax.get_xlim()
ax.axvspan(xlim[0], MIN_RATIO, alpha=0.12, color="#d62728")
ax.axvspan(MAX_RATIO, xlim[1], alpha=0.12, color="#d62728")
ax.text(
    (xlim[0] + MIN_RATIO) / 2, ax.get_ylim()[1] * 0.82, "drop\n(too tall)", ha="center", color="#d62728", fontsize=9
)
ax.text(
    (MAX_RATIO + xlim[1]) / 2, ax.get_ylim()[1] * 0.82, "drop\n(too wide)", ha="center", color="#d62728", fontsize=9
)

ax.set_xlabel("Aspect ratio  (width / height)")
ax.set_ylabel("Number of images")
ax.set_title(f"Image aspect ratio distribution  ({len(image_rows)} images)")
ax.legend()
plt.tight_layout()
plt.show()

n_will_drop = int((~image_rows["ratio"].between(MIN_RATIO, MAX_RATIO)).sum())
print(f"Will drop: {n_will_drop} / {len(image_rows)} images  ({100 * n_will_drop / len(image_rows):.1f}%)")

### What does a "bad" image actually look like?

In [ ]:
to_drop = image_rows[~image_rows["ratio"].between(MIN_RATIO, MAX_RATIO)].copy()
to_drop["reason"] = to_drop["ratio"].apply(lambda r: "too tall" if r < MIN_RATIO else "too wide")

if to_drop.empty:
    print("No images to drop with these bounds.")
else:
    print(f"{len(to_drop)} images will be removed:")
    NCOLS = min(5, len(to_drop))
    nrows = (len(to_drop) + NCOLS - 1) // NCOLS
    fig, axes = plt.subplots(nrows, NCOLS, figsize=(NCOLS * 2.5, nrows * 3.2))
    axes = axes.flatten() if nrows * NCOLS > 1 else [axes]

    for i, (_, row) in enumerate(to_drop.iterrows()):
        ax = axes[i]
        try:
            img = PILImage.open(BytesIO(bytes(row["binary_content"])))
            img.thumbnail((128, 256), RESAMPLE)
            ax.imshow(img)
        except (OSError, ValueError):
            ax.text(0.5, 0.5, "err", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(
            f"r = {row['ratio']:.2f}  ({row['reason']})\n{int(row['width'])} x {int(row['height'])} px",
            fontsize=7,
            color="#d62728",
        )
        ax.axis("off")

    for i in range(len(to_drop), len(axes)):
        axes[i].axis("off")

    plt.suptitle("Images that will be removed by the aspect-ratio filter", fontsize=10)
    plt.tight_layout()
    plt.show()

### Apply the filter

In [ ]:
filter_stage = InterleavedAspectRatioFilterStage(min_aspect_ratio=MIN_RATIO, max_aspect_ratio=MAX_RATIO)
filtered_batch = filter_stage.process(batch)

print(f"{'':30s} {'rows':>6}  {'images':>7}  {'text':>6}  {'samples':>8}")
print(f"  {'-' * 60}")
for label, b in [("Original", batch), ("After image filter", filtered_batch)]:
    print(
        f"  {label:<28} {b.count():6d}  {b.count(modality='image'):7d}  "
        f"{b.count(modality='text'):6d}  {b.num_items:8d}"
    )
dropped = batch.count() - filtered_batch.count()
print(f"  {'Dropped':28}  {dropped:5d} rows")

### Before & after — one document up close

Below we pick a sample that had at least one image removed and render its image rows
before and after filtering. Notice that the positions are automatically renumbered.

In [ ]:
df_after = filtered_batch.to_pandas()

imgs_before = df[df["modality"] == "image"].groupby("sample_id").size()
imgs_after = df_after[df_after["modality"] == "image"].groupby("sample_id").size()

# Find a sample that lost at least one image
focus_id = None
for sid in imgs_before.index:
    before_n = int(imgs_before[sid])
    after_n = int(imgs_after.get(sid, 0))
    if before_n > after_n:
        focus_id = sid
        before_n_img = before_n
        after_n_img = after_n
        break

if focus_id is None:
    print("No sample lost images with these filter bounds.")
else:
    ipy_display(Markdown(f"#### Sample `{focus_id}`"))
    print(f"  Images: {before_n_img} before  \u2192  {after_n_img} after  ({before_n_img - after_n_img} removed)")
    print()
    render_image_strip(df[df["sample_id"] == focus_id], f"Before filter  ({before_n_img} images)")
    render_image_strip(
        df_after[df_after["sample_id"] == focus_id], f"After filter   ({after_n_img} images, positions renumbered)"
    )

---
## Section 4 — Filter: Text Quality

Many MINT-1T text rows are very short — page headers, figure captions like "Figure 1.",
or single words from OCR. We drop them with a **custom filter**.

**The `BaseInterleavedFilterStage` API** — subclass it and implement one method:

```python
def content_keep_mask(self, task: InterleavedBatch, df: pd.DataFrame) -> pd.Series:
    ...  # return bool Series: True = keep the row, False = drop it
```

The base class handles the rest automatically:
- **Metadata rows are never passed** to your mask — only `"text"` and `"image"` rows
- **Positions are renumbered** (0, 1, 2 …) after your mask is applied
- **Orphan metadata rows** (samples that lost all content) are removed

In [ ]:
text_rows = df[df["modality"] == "text"].copy()
text_rows["char_count"] = text_rows["text_content"].fillna("").str.len()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(text_rows["char_count"].clip(0, 600), bins=50, color="#4C72B0", edgecolor="white", alpha=0.85)
ax.axvline(MIN_CHARS, color="#d62728", linestyle="--", linewidth=2, label=f"min_chars = {MIN_CHARS}")
xlim = ax.get_xlim()
ax.axvspan(xlim[0], MIN_CHARS, alpha=0.12, color="#d62728")
ax.text((xlim[0] + MIN_CHARS) / 2, ax.get_ylim()[1] * 0.82, "drop", ha="center", color="#d62728", fontsize=9)
ax.set_xlabel("Text length (characters, clipped at 600)")
ax.set_ylabel("Number of text rows")
ax.set_title(f"Text length distribution  ({len(text_rows)} text rows)")
ax.legend()
plt.tight_layout()
plt.show()

n_short = int((text_rows["char_count"] < MIN_CHARS).sum())
print(f"Short text rows (< {MIN_CHARS} chars): {n_short} / {len(text_rows)}  ({100 * n_short / len(text_rows):.1f}%)")

In [ ]:
@dataclass
class ShortTextFilterStage(BaseInterleavedFilterStage):
    """Drop text rows shorter than min_chars characters."""

    min_chars: int = MIN_CHARS
    name: str = "short_text_filter"

    def content_keep_mask(self, _task: InterleavedBatch, df: pd.DataFrame) -> pd.Series:
        keep = pd.Series(True, index=df.index, dtype=bool)
        is_text = df["modality"] == "text"
        too_short = is_text & (df["text_content"].fillna("").str.len() < self.min_chars)
        keep[too_short] = False
        return keep


final_batch = ShortTextFilterStage(min_chars=MIN_CHARS).process(filtered_batch)

# Apply the same criterion to df_filtered to find what was dropped.
# (Position renumbering means we cannot use set-diff on (sample_id, position).)
df_filtered = filtered_batch.to_pandas()
is_text = df_filtered["modality"] == "text"
too_short = df_filtered["text_content"].fillna("").str.len() < MIN_CHARS
dropped_text = df_filtered[is_text & too_short]

print(f"Dropped {len(dropped_text)} short text rows (< {MIN_CHARS} chars). Examples:")
print()
for _, row in dropped_text.head(10).iterrows():
    tc = row["text_content"]
    s = str(tc) if tc is not None and not pd.isna(tc) else ""
    print(f"  [{len(s):3d} chars]  {s!r}")

In [ ]:
print("Pipeline summary:")
print(f"  {'':33} {'rows':>6}  {'images':>7}  {'text':>6}  {'samples':>8}")
print(f"  {'-' * 65}")
stages = [
    ("Original (WebDataset read)", batch),
    ("After image filter (ratio 0.5-2)", filtered_batch),
    (f"After text filter  (>= {MIN_CHARS} chars)", final_batch),
]
for label, b in stages:
    print(
        f"  {label:<33} {b.count():6d}  {b.count(modality='image'):7d}  "
        f"{b.count(modality='text'):6d}  {b.num_items:8d}"
    )

---
## Section 5 — Save & Verify

NeMo Curator provides two writer stages for interleaved data:
- **`InterleavedParquetWriterStage`** — columnar format, efficient for downstream ML training
- **`InterleavedWebdatasetWriterStage`** — writes back to MINT-1T-compatible tar shards

Both share the same `on_materialize_error` and `schema_overrides` parameters.

### Writing to Parquet

`on_materialize_error` controls what happens when a lazy image fetch fails at write time:

| Mode | Behavior |
|------|----------|
| `"error"` | Raise immediately — safest for debugging |
| `"warn"` | Keep the row, log a warning, `binary_content` stays null |
| `"drop_row"` | Drop only the failed row |
| `"drop_sample"` | Drop the entire sample if any row fails |

`schema_overrides` pins PyArrow types for passthrough columns, preventing type
inconsistencies when merging files from multiple pipeline runs.

In [ ]:
output_dir = tempfile.mkdtemp(prefix="nemo_curator_tutorial_")

writer = InterleavedParquetWriterStage(
    path=output_dir,
    materialize_on_write=True,  # fetch any remaining lazy binary_content at write time
    mode="overwrite",
    on_materialize_error="drop_row",  # recommended for production
    schema_overrides={
        "url": pa.large_string(),
        "pdf_name": pa.large_string(),
        "image_metadata": pa.large_string(),
    },
)
write_result = writer.process(final_batch)
parquet_path = write_result.data[0]

meta = pq.read_metadata(parquet_path)
print(f"Written: {parquet_path}")
print(f"  {meta.num_rows} rows  |  columns: {pq.read_schema(parquet_path).names}")

In [ ]:
file_task = FileGroupTask(dataset_name="mint1t", data=[parquet_path], _metadata={})

pq_reader = InterleavedParquetReaderStage(
    schema_overrides={
        "url": pa.large_string(),
        "pdf_name": pa.large_string(),
        "image_metadata": pa.large_string(),
    }
)
roundtrip = pq_reader.process(file_task)
if isinstance(roundtrip, list):
    roundtrip = roundtrip[0]

orig_ids = set(final_batch.to_pandas()["sample_id"].tolist())
rt_ids = set(roundtrip.to_pandas()["sample_id"].tolist())

print(f"Roundtrip: {roundtrip.num_items} samples, {roundtrip.count()} rows")
print(f"  Images with binary_content: {roundtrip.count(modality='image')}")
print(f"  Sample IDs match original:  {orig_ids == rt_ids}")

### Render the same document from Parquet

We render the same sample we explored in Section 2 — now read back from Parquet. The
images come from `binary_content` stored in the file rather than the original tar.

In [ ]:
rt_df = roundtrip.to_pandas()

# Render the same sample from Section 2; fall back if it was filtered out
render_id = rich_sample_id if rich_sample_id in rt_ids else next(iter(sorted(rt_ids)))
if render_id != rich_sample_id:
    ipy_display(Markdown(f"> Note: `rich_sample_id` was filtered out; rendering `{render_id}` instead."))

render_sample(rt_df, render_id)

In [ ]:
# Write to WebDataset tar (MINT-1T format)
wds_dir = tempfile.mkdtemp(prefix="nemo_curator_wds_")
wds_writer = InterleavedWebdatasetWriterStage(
    path=wds_dir, materialize_on_write=True, mode="overwrite", on_materialize_error="drop_row"
)
tar_path = wds_writer.process(final_batch).data[0]

with tarfile.open(tar_path) as tf:
    members = tf.getmembers()
json_count = sum(1 for m in members if m.name.endswith(".json"))
print(f"Written: {tar_path}")
print(f"  {json_count} JSON members (one per sample)  +  {len(members) - json_count} image members")

# Read back and verify
wds_task = FileGroupTask(dataset_name="mint1t_out", data=[tar_path], _metadata={})
wds_back = InterleavedWebdatasetReaderStage(
    materialize_on_read=False, per_image_fields=["image_metadata"], sample_id_field="sample_id"
).process(wds_task)
if isinstance(wds_back, list):
    wds_back = wds_back[0]

wds_ids = set(wds_back.to_pandas()["sample_id"].tolist())
print(f"\nWebDataset roundtrip: {wds_back.num_items} samples, {wds_back.count()} rows")
print(f"Sample IDs match: {wds_ids == orig_ids}")

---
## Next Steps

| Goal | Where to look |
|------|---------------|
| Scale to many shards (Ray cluster) | `tutorials/interleaved/getting-started/interleaved_pipeline.py` |
| Distributed execution | `Pipeline` + `RayClient` in `nemo_curator/pipeline/` |
| Write a custom annotator (score / tag rows) | Subclass `BaseInterleavedAnnotatorStage` in `nemo_curator/stages/interleaved/stages.py` |
| Schema control API | `nemo_curator/stages/interleaved/utils/schema.py` |
| Architecture & source_ref internals | `nemo_curator/stages/interleaved/README.md` |
| PDF processing (Nemotron-Parse) | `nemo_curator/stages/interleaved/pdf/nemotron_parse/` |